🟢 Débutant | ⏱ 80 min | 🔑 base de données, table, SQL, sqlite3, INSERT, SELECT, UPDATE, DELETE, `?`

# Module 12 — Ranger les données en base : SQLite

> 🎬 **Regarde d'abord la vidéo du module** (scénario `python-debutant-12`, à venir).

**Ce que tu sais déjà**
- Module 06 : les **tuples** `(a, b)` et le déballage `a, b = ...` ; la liste de dictionnaires = une table.
- Module 07 : écrire des **fonctions** avec paramètres.
- Module 08 : lire et écrire des fichiers, `try/except`.
- Module 10 : pandas, `to_dict("records")`, `groupby`.
- Module 11 : les **objets**, leurs attributs et leurs méthodes (`connexion.execute(...)` en sera une !).

**Ce que tu vas apprendre**
- Ce qu'est une **base de données**, une **table**, et le langage **SQL**.
- Créer une base SQLite avec le module `sqlite3` : `connect`, `execute`, `CREATE TABLE`, `commit`.
- Ajouter des lignes avec `INSERT` et des **requêtes paramétrées** (`?`), et comprendre pourquoi c'est obligatoire.
- Interroger la base avec `SELECT … WHERE … ORDER BY` et `fetchall`, la modifier avec `UPDATE` et `DELETE`.
- Récupérer une requête directement dans un DataFrame avec `pd.read_sql`.

**Comment lire ce notebook**
- Exécute les cellules **dans l'ordre**, une par une, avec `Shift + Entrée`. Ici, l'ordre compte encore plus :
  la base est créée au début et les cellules suivantes s'en servent.
- Les cellules 🔮 te demandent de **prédire** le résultat avant d'exécuter : écris ta réponse, puis vérifie.
- Les cellules ⚠️ plantent **exprès** : c'est pour apprendre à lire le message d'erreur.
- Pour un 🔮 dont tu ne peux pas calculer le chiffre exact (tu n'as pas toutes les données sous les yeux), prédis la **forme** du résultat : un nombre, un texte, un tableau, combien de lignes. C'est ce qui compte.
- En cas de doute, redémarre le kernel (Jupyter : *Kernel → Restart* ; VS Code : bouton *Restart* en haut du notebook) et réexécute **tout depuis le début** : la base est
  recréée à neuf à chaque fois.

## 1. Pourquoi une base de données ?

Jusqu'ici, les relevés vivaient dans un **fichier CSV** : du texte, ligne après ligne. C'est simple, mais :
- pour trouver les relevés de Toulouse, il faut **relire tout le fichier** ;
- pour corriger une valeur, il faut **réécrire tout le fichier** ;
- deux personnes qui écrivent en même temps risquent de s'écraser.

Une **base de données**, c'est un **classeur géré par un gardien** : tu ne touches pas aux feuilles toi-même, tu
demandes au gardien « range ça », « donne-moi les relevés de Toulouse », « corrige cette case ». Il range tout
proprement et retrouve vite.

Le vocabulaire :
- une **table** est une feuille du classeur : des **colonnes** avec un nom et un type, et des **lignes** ;
- **SQL** (*Structured Query Language*, langage de requêtes structurées) est la langue qu'on parle au gardien. Une
  **requête** est une phrase en SQL : `SELECT date FROM releves WHERE ville = 'Albi'` ;
- **SQLite** est une base de données qui tient dans **un seul fichier** (ici `meteolocale.db`). Python sait lui parler
  avec le module standard `sqlite3` : rien à installer.

Les mots-clés SQL s'écrivent en **MAJUSCULES** par convention (`SELECT`, `FROM`, `WHERE`) : on les repère d'un coup
d'œil au milieu des noms de colonnes en minuscules.

## 2. Ouvrir la base : `connect`

`sqlite3.connect("chemin.db")` ouvre la base. **Si le fichier n'existe pas, il est créé.** Le résultat est un
**objet connexion** (module 11) : c'est notre ligne directe avec le gardien. On lui parle avec sa méthode `execute`,
et on raccroche avec sa méthode `close()`.

Comme tout ce qu'on écrit, la base va dans `sorties/`.

In [ ]:
import sqlite3
from pathlib import Path

Path("sorties").mkdir(exist_ok=True)
connexion = sqlite3.connect("sorties/meteolocale.db")
print(type(connexion))
print(Path("sorties/meteolocale.db").exists())
connexion.close()

## 3. Créer une table : `CREATE TABLE`

Avant de ranger des relevés, il faut dessiner la feuille : le **nom** de la table et ses **colonnes**, chacune avec
un **type SQL** :

| Type SQL | Type Python | Exemple |
|---|---|---|
| `TEXT` | `str` | `"2026-09-01"`, `"Toulouse"` |
| `REAL` | `float` | `25.9` |
| `INTEGER` | `int` | `11` |

Juste avant, `DROP TABLE IF EXISTS releves` **supprime** la table si elle existe déjà (*drop* = jeter). Ainsi, à
chaque exécution du notebook, on **repart d'une base neuve** : les mêmes cellules donnent toujours le même résultat.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
connexion.commit()
connexion.close()
print("Table releves créée")

`connexion.execute("...")` envoie une requête SQL au gardien. La requête est une **chaîne de caractères** Python :
entre guillemets. `connexion.commit()` **valide** les changements (on y revient à la section 4).

⚠️ **Erreur fréquente : créer deux fois la même table.**

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT)")

**Lisons le message.** Dernière ligne : `OperationalError: table releves already exists`.

- `OperationalError` : une erreur **SQL**, renvoyée par le gardien (le module `sqlite3` la rend à Python).
- Traduction : « la table releves existe déjà ». La cellule précédente l'a créée, le fichier s'en souvient.
- Le type complet s'écrit `sqlite3.OperationalError` : c'est un type d'erreur rangé dans le module `sqlite3`.

**Correction** : mettre `DROP TABLE IF EXISTS releves` avant le `CREATE TABLE` (comme à la cellule d'avant), pour
repartir de zéro à chaque exécution.

La cellule qui a planté avait ouvert une connexion (la ligne `connexion = sqlite3.connect(...)` s'est exécutée
avant l'erreur). La variable `connexion` existe donc toujours : on ferme **cette** connexion-là, pour repartir propre.

In [ ]:
connexion.close()

## 4. Ajouter des lignes : `INSERT` et les requêtes paramétrées `?`

Pour ajouter une ligne : `INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)`. Chaque `?` est une **place réservée** ;
les vraies valeurs arrivent à part, dans un **tuple** (module 06), donné en deuxième argument de `execute`. Le
gardien range la première valeur à la place du premier `?`, et ainsi de suite. Six colonnes, six `?`, six valeurs.

Puis `connexion.commit()` **valide** : tant qu'on n'a pas fait `commit`, les changements restent **en brouillon**.
C'est le bouton « Enregistrer » du classeur.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute(
    "INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)",
    ("2026-09-01", "Toulouse", 12.0, 25.9, 17.4, 11),
)
connexion.commit()

lignes = connexion.execute("SELECT * FROM releves").fetchall()
print(lignes)
connexion.close()

`SELECT * FROM releves` demande **toutes** les colonnes (`*`) de **toutes** les lignes, et `fetchall()` les rapporte
sous forme de **liste de tuples** : une ligne = un tuple. On détaille `SELECT` à la section 5.

La requête est un peu longue : on l'a écrite **sur plusieurs lignes**. Entre des parenthèses, Python accepte qu'on
passe à la ligne : c'est plus lisible.

### Pourquoi `?` et jamais une f-string ?

Tu pourrais être tenté d'écrire la valeur directement dans la requête avec une f-string (module 02). **Ne le fais
jamais.** Deux raisons, très concrètes.

**Raison 1 : une apostrophe casse tout.** En SQL, le texte s'écrit entre apostrophes `'...'`. Si la valeur contient
elle-même une apostrophe, comme la commune de L'Isle-Jourdain, la requête devient illisible pour le gardien.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
ville = "L'Isle-Jourdain"
requete = f"SELECT * FROM releves WHERE ville = '{ville}'"
print(requete)
connexion.execute(requete)

**Lisons le message.** `print` a d'abord montré la requête fabriquée :
`SELECT * FROM releves WHERE ville = 'L'Isle-Jourdain'`. Dernière ligne :
`OperationalError: near "Isle": syntax error`.

- « erreur de syntaxe près de "Isle" » : pour le gardien, le texte `'L'` s'arrête à la deuxième apostrophe, puis
  arrive un mot `Isle` qu'il ne comprend pas.
- Ce n'est pas ton Python qui est faux : c'est la **requête fabriquée** qui l'est.

**Raison 2 : la sécurité.** Imagine que la ville vienne d'un formulaire web (comme `input()` au module 02), et
qu'une personne mal intentionnée tape : `x' OR '1'='1`. La f-string fabrique alors
`SELECT * FROM releves WHERE ville = 'x' OR '1'='1'` : la condition `'1'='1'` est toujours vraie, et la requête
renvoie **toutes** les lignes. Avec `DELETE` au lieu de `SELECT`, elle effacerait tout. Cette attaque s'appelle
une **injection SQL** ; c'est l'une des failles les plus courantes du web.

**Avec `?`, ces deux problèmes disparaissent** : la valeur voyage **à part**, elle n'est jamais lue comme du SQL. Le
gardien la range telle quelle, apostrophes comprises.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
ville = "L'Isle-Jourdain"
print(connexion.execute("SELECT * FROM releves WHERE ville = ?", (ville,)).fetchall())

piege = "x' OR '1'='1"
print(connexion.execute("SELECT * FROM releves WHERE ville = ?", (piege,)).fetchall())
connexion.close()

Deux listes vides, et **aucune erreur** : aucune ville ne s'appelle ainsi, et le piège est traité comme un simple
nom (bizarre) de ville. La règle du module : **une valeur dans une requête = un `?`**. Toujours.

Remarque la virgule dans `(ville,)` : c'est un **tuple à un seul élément**. Sans la virgule, `(ville)` n'est
qu'une chaîne entre parenthèses. L'erreur ci-dessous montre ce qui se passe alors.

⚠️ **Erreur fréquente : oublier la virgule du tuple à un élément.**

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("SELECT * FROM releves WHERE ville = ?", ("Toulouse"))

**Lisons le message.** Dernière ligne :
`ProgrammingError: Incorrect number of bindings supplied. The current statement uses 1, and there are 8 supplied.`

- `ProgrammingError` : une erreur dans la façon d'appeler `sqlite3`.
- Traduction : « nombre de valeurs incorrect. La requête en utilise 1 (un seul `?`), et on en a fourni 8. »
- 8 ? C'est le nombre de **lettres** de `"Toulouse"`. Sans virgule, Python a donné une chaîne, que `sqlite3` a
  parcourue lettre par lettre (comme une boucle `for` sur une chaîne, module 04).

**Correction** : `("Toulouse",)`, avec la virgule.

La cellule qui a planté avait ouvert une connexion (la ligne `connexion = sqlite3.connect(...)` s'est exécutée
avant l'erreur). La variable `connexion` existe donc toujours : on ferme **cette** connexion-là, pour repartir propre.

In [ ]:
connexion.close()

⚠️ **Erreur fréquente : un `execute` dans un `execute`.** En complétant un `connexion.execute(...)`, on a
parfois envie de remplacer les `...` par l'appel **entier**. Résultat :

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute(connexion.execute("SELECT * FROM releves"))

**Lisons le message.** Dernière ligne : `TypeError: execute() argument 1 must be str, not sqlite3.Cursor`.

- « l'argument 1 de `execute()` doit être une chaîne (`str`), pas un `sqlite3.Cursor` ».
- Un **curseur** (*cursor*), c'est ce que **renvoie** `execute` (on le détaille à la section 5). Ici, l'`execute`
  intérieur a renvoyé un curseur, qui a été donné à l'`execute` extérieur à la place de la requête.
- `execute` attend en premier argument la **requête**, entre guillemets, et en deuxième le tuple des valeurs.

**Correction** : un seul `execute`, avec la chaîne SQL dedans : `connexion.execute("SELECT * FROM releves")`.
Quand un starter montre `connexion.execute(...)`, les `...` remplacent **seulement** ce qui va entre les parenthèses.

La cellule qui a planté avait ouvert une connexion (la ligne `connexion = sqlite3.connect(...)` s'est exécutée
avant l'erreur). La variable `connexion` existe donc toujours : on ferme **cette** connexion-là, pour repartir propre.

In [ ]:
connexion.close()

### Le piège du `commit` oublié

Sans `commit`, les changements restent un **brouillon** visible seulement par ta connexion. Une **autre**
connexion (un collègue, ou un autre programme) ne voit rien. Et si tu fermes sans valider, le brouillon est perdu.

In [ ]:
import sqlite3

ecriture = sqlite3.connect("sorties/meteolocale.db")
ecriture.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", ("2026-09-01", "Albi", 12.6, 26.8, 9.4, 18))
print("Ma connexion voit :", len(ecriture.execute("SELECT * FROM releves").fetchall()), "lignes")

collegue = sqlite3.connect("sorties/meteolocale.db")
print("Le collègue voit   :", len(collegue.execute("SELECT * FROM releves").fetchall()), "ligne")
collegue.close()

ecriture.close()          # fermeture SANS commit : la ligne d'Albi est perdue
verif = sqlite3.connect("sorties/meteolocale.db")
print("Après fermeture    :", len(verif.execute("SELECT * FROM releves").fetchall()), "ligne")
verif.close()

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", ("2026-09-01", "Nîmes", 15.1, 29.2, 10.0, 42))
connexion.commit()
connexion.close()

autre = sqlite3.connect("sorties/meteolocale.db")
print(len(autre.execute("SELECT * FROM releves").fetchall()))
autre.close()

<details><summary>👀 Voir l'explication</summary>

`2`. La ligne de Toulouse (validée à la section 4) et celle de Nîmes, validée par `commit` avant de fermer. La ligne
d'Albi de la cellule d'avant, jamais validée, n'existe pas.

</details>

### Ranger tout le fichier : une boucle d'`INSERT`

Pour les 180 relevés, on combine ce qu'on sait : pandas lit le CSV, `to_dict("records")` donne une liste de
dictionnaires (module 10), et une boucle fait un `INSERT` par ligne. **Un seul `commit` à la fin** suffit.

Comme cette préparation va resservir, on la range dans une **fonction** (module 07) qui renvoie la connexion.

In [ ]:
import sqlite3
import pandas as pd


def creer_base(chemin):
    connexion = sqlite3.connect(chemin)
    connexion.execute("DROP TABLE IF EXISTS releves")
    connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
    for f in pd.read_csv("../donnees/meteo.csv").to_dict("records"):
        connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", (f["date"], f["ville"], f["temp_min"], f["temp_max"], f["pluie_mm"], f["vent_kmh"]))
    connexion.commit()
    return connexion


connexion = creer_base("sorties/meteolocale.db")
print(len(connexion.execute("SELECT * FROM releves").fetchall()), "relevés en base")
connexion.close()

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
releve = {"date": "2026-09-02", "ville": "Tarbes", "temp_min": 12.5, "temp_max": 23.1, "pluie_mm": 8.1, "vent_kmh": 6}
valeurs = (releve["date"], releve["ville"], releve["temp_max"])
print(len(valeurs), valeurs[1])

<details><summary>👀 Voir l'explication</summary>

`3 Tarbes`. Le tuple contient 3 valeurs ; la position 1 est la deuxième : la ville. Pour une requête à 3 `?`, ce
tuple aurait exactement la bonne taille.

</details>

## 5. Interroger : `SELECT … FROM … WHERE … ORDER BY`

Une requête `SELECT` se lit comme une phrase :

```sql
SELECT date, temp_max      -- quelles colonnes je veux
FROM releves               -- dans quelle table
WHERE ville = ?            -- quelles lignes (la condition)
ORDER BY temp_max DESC     -- dans quel ordre (DESC = décroissant)
```

- `execute(...)` renvoie un **curseur** : un objet qui pointe sur le résultat, comme un marque-page.
- `curseur.fetchall()` (*fetch all* = « va tout chercher ») rapporte toutes les lignes dans une **liste de tuples**.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
curseur = connexion.execute("SELECT date, temp_max FROM releves WHERE ville = ?", ("Albi",))
lignes = curseur.fetchall()
print(len(lignes), "lignes pour Albi")
print(lignes[:3])
connexion.close()

Chaque ligne est un tuple `(date, temp_max)`, dans l'ordre des colonnes demandées. On le **déballe** comme au
module 06 :

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
lignes = connexion.execute("SELECT date, temp_max FROM releves WHERE ville = ?", ("Albi",)).fetchall()
for date, temp_max in lignes[:3]:
    print(f"{date} : {temp_max} °C")
connexion.close()

### Conditions et tri

- `WHERE` accepte `=`, `!=` (ou `<>`), `>`, `>=`, `<`, `<=`. Attention : en SQL, l'égalité s'écrit avec **un seul** `=`.
- On combine avec `AND` (et) et `OR` (ou), en toutes lettres.
- `ORDER BY colonne` trie du plus petit au plus grand ; `ORDER BY colonne DESC` du plus grand au plus petit.
- Une requête peut contenir **plusieurs** `?` : le tuple donne les valeurs **dans l'ordre**.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
requete = "SELECT date, pluie_mm FROM releves WHERE ville = ? AND pluie_mm > ? ORDER BY pluie_mm DESC"
lignes = connexion.execute(requete, ("Toulouse", 10)).fetchall()
print(lignes)
connexion.close()

🔮 **Prédis** : Avant d'exécuter : `lignes` sera-t-elle une liste ou un tuple ? Que contiendra chacun de ses éléments ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
lignes = connexion.execute(
    "SELECT ville, pluie_mm FROM releves WHERE date = ? AND pluie_mm > ?",
    ("2026-09-01", 15),
).fetchall()
print(lignes)
connexion.close()

<details><summary>👀 Voir l'explication</summary>

**Une liste de tuples**, un tuple `(ville, pluie_mm)` par ligne trouvée. Ici `[('Toulouse', 17.4)]` : le
1er septembre, une seule ville dépasse 15 mm. Le résultat reste une **liste** (d'un seul tuple) : `fetchall` renvoie
toujours une liste, même vide, même avec une seule ligne.

</details>

🔮 **Prédis** : Avant d'exécuter : combien de lignes ce jour-là ? La première sera-t-elle la ville la plus chaude ou la plus froide ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
lignes = connexion.execute(
    "SELECT ville, temp_max FROM releves WHERE date = ? ORDER BY temp_max DESC",
    ("2026-09-01",),
).fetchall()
print(len(lignes), lignes[0])
connexion.close()

<details><summary>👀 Voir l'explication</summary>

**6 lignes**, une par ville, et la première est **la plus chaude** : `DESC` trie du plus grand au plus petit. Ici
`6 ('Montpellier', 29.2)` (ou Nîmes, qui a aussi 29.2 ce jour-là).

</details>

🔮 **Prédis** : Avant d'exécuter : la deuxième chose affichée, `lignes[0]`, sera-t-elle un nombre, une liste ou un tuple ? Avec combien de valeurs ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
lignes = connexion.execute("SELECT * FROM releves WHERE vent_kmh >= ?", (55,)).fetchall()
print(len(lignes))
print(lignes[0])
connexion.close()

<details><summary>👀 Voir l'explication</summary>

**Un tuple à 6 valeurs** : `SELECT *` rapporte toutes les colonnes, dans l'ordre de la table (date, ville, temp_min,
temp_max, pluie_mm, vent_kmh). Avant lui, le nombre de lignes trouvées (ici `3`).

</details>

⚠️ **Erreur fréquente : un nom de colonne ou de table mal écrit.**

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("SELECT date, temp_maxi FROM releves").fetchall()

**Lisons le message.** Dernière ligne : `OperationalError: no such column: temp_maxi`.

- « pas de colonne de ce nom : temp_maxi ». Le gardien cite le nom fautif.
- Même famille : `no such table: releve` si le nom de la **table** est faux (ici il s'appelle `releves`, avec un s).
- C'est l'équivalent SQL du `KeyError` de pandas (module 10).

**Correction** : `temp_max`. En cas de doute, `SELECT * FROM releves` montre l'ordre des colonnes, et le
`CREATE TABLE` de la section 3 donne leurs noms.

La cellule qui a planté avait ouvert une connexion (la ligne `connexion = sqlite3.connect(...)` s'est exécutée
avant l'erreur). La variable `connexion` existe donc toujours : on ferme **cette** connexion-là, pour repartir propre.

In [ ]:
connexion.close()

## 6. Modifier : `UPDATE … SET … WHERE`

La station de Tarbes signale une erreur : le 2 septembre, il est tombé 8.6 mm, pas 8.1. On corrige **une case**
sans réécrire le fichier :

```sql
UPDATE releves SET pluie_mm = ? WHERE ville = ? AND date = ?
```

« Mets à jour la table `releves` : **mets** `pluie_mm` à cette valeur, **là où** la ville et la date sont celles-ci. »
Puis `commit`.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("UPDATE releves SET pluie_mm = ? WHERE ville = ? AND date = ?", (8.6, "Tarbes", "2026-09-02"))
connexion.commit()

print(connexion.execute("SELECT * FROM releves WHERE ville = ? AND date = ?", ("Tarbes", "2026-09-02")).fetchall())
connexion.close()

⚠️ **Le `WHERE` est vital.** `UPDATE releves SET pluie_mm = 8.6` **sans** `WHERE` met 8.6 mm dans **les 180
lignes**. Aucune erreur, aucun avertissement : le gardien obéit. Démonstration sur une petite base de test, pour
ne pas abîmer la vraie :

In [ ]:
import sqlite3

test = sqlite3.connect("sorties/test_sans_where.db")
test.execute("DROP TABLE IF EXISTS releves")
test.execute("CREATE TABLE releves (ville TEXT, pluie_mm REAL)")
for ville, pluie in [("Albi", 9.4), ("Tarbes", 8.1), ("Nîmes", 0.0)]:
    test.execute("INSERT INTO releves VALUES (?, ?)", (ville, pluie))
test.commit()

test.execute("UPDATE releves SET pluie_mm = ?", (8.6,))     # oubli du WHERE !
test.commit()
print(test.execute("SELECT * FROM releves").fetchall())
test.close()

Les trois villes ont maintenant 8.6 mm. Réflexe de pro : **écris le `WHERE` en premier**, et vérifie-le avec un
`SELECT … WHERE …` avant de lancer l'`UPDATE`.

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("UPDATE releves SET vent_kmh = ? WHERE ville = ? AND date = ?", (40, "Albi", "2026-09-01"))
connexion.commit()
lignes = connexion.execute("SELECT vent_kmh FROM releves WHERE ville = ?", ("Albi",)).fetchall()
print(lignes[0], len(lignes))
connexion.close()

<details><summary>👀 Voir l'explication</summary>

`(40,) 30`. Seule la ligne d'Albi du 1er septembre a changé (18 → 40). Albi a toujours 30 lignes. Remarque `(40,)` :
une ligne d'**une seule** colonne reste un tuple, à un élément.

</details>

## 7. Supprimer : `DELETE FROM … WHERE`

Pour enlever des lignes : `DELETE FROM releves WHERE condition`, puis `commit`. Par exemple, une ligne de test
glissée par erreur :

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", ("2026-09-31", "Test", 0.0, 0.0, 0.0, 0))
connexion.commit()
print("Avant :", len(connexion.execute("SELECT * FROM releves").fetchall()))

connexion.execute("DELETE FROM releves WHERE ville = ?", ("Test",))
connexion.commit()
print("Après :", len(connexion.execute("SELECT * FROM releves").fetchall()))
connexion.close()

Même avertissement qu'avec `UPDATE`, en pire : `DELETE FROM releves` **sans** `WHERE` vide la table entière. Il n'y
a pas de corbeille. Écris le `WHERE` d'abord, teste-le avec un `SELECT`.

🔮 **Prédis** : Avant d'exécuter : le nombre affiché sera-t-il plus petit, égal ou plus grand que 30 ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import sqlite3

connexion = sqlite3.connect("sorties/meteolocale.db")
connexion.execute("DELETE FROM releves WHERE ville = ? AND pluie_mm > ?", ("Perpignan", 0))
connexion.commit()
print(len(connexion.execute("SELECT * FROM releves WHERE ville = ?", ("Perpignan",)).fetchall()))
connexion.close()

<details><summary>👀 Voir l'explication</summary>

**Plus petit que 30** : Perpignan avait 30 lignes, et on a supprimé ses jours **pluvieux** ; il reste ses jours secs
(ici `9`). (On recrée la base complète juste après, ne
t'inquiète pas.)

</details>

La cellule suivante utilise la fonction `creer_base`, écrite à la section 4. Si tu as redémarré le kernel entre-temps,
tu verras `NameError: name 'creer_base' is not defined` : réexécute d'abord la cellule de la section 4.

In [ ]:
connexion = creer_base("sorties/meteolocale.db")      # on remet les 180 relevés d'origine
print(len(connexion.execute("SELECT * FROM releves").fetchall()), "relevés")
connexion.close()

## 8. De la base à pandas : `pd.read_sql`

Une liste de tuples, c'est bien ; un DataFrame, c'est mieux pour calculer (module 10). `pd.read_sql(requete,
connexion)` exécute la requête et renvoie **directement un DataFrame**, avec les noms de colonnes. Les valeurs
passent toujours par des `?`, données avec l'argument nommé `params`.

In [ ]:
import sqlite3
import pandas as pd

connexion = sqlite3.connect("sorties/meteolocale.db")
toulouse = pd.read_sql("SELECT * FROM releves WHERE ville = ?", connexion, params=("Toulouse",))
connexion.close()

print(toulouse.shape)
print(round(toulouse["temp_max"].mean(), 1))
toulouse.head(3)

Le meilleur des deux mondes : la base **range et retrouve** (`WHERE`), pandas **calcule** (`groupby`, `mean`).

In [ ]:
import sqlite3
import pandas as pd

connexion = sqlite3.connect("sorties/meteolocale.db")
tout = pd.read_sql("SELECT ville, temp_max, pluie_mm FROM releves", connexion)
connexion.close()

tout.groupby("ville")[["temp_max", "pluie_mm"]].mean().round(1)

🔮 **Prédis** : Avant d'exécuter : que va afficher cette cellule ? Écris ta réponse sur papier (ou dans ta tête), puis exécute.

In [ ]:
import sqlite3
import pandas as pd

connexion = sqlite3.connect("sorties/meteolocale.db")
pluvieux = pd.read_sql("SELECT * FROM releves WHERE pluie_mm > ?", connexion, params=(15,))
connexion.close()
print(pluvieux.shape[1], len(set(pluvieux["ville"])) <= 6)

<details><summary>👀 Voir l'explication</summary>

`6 True`. `SELECT *` rapporte les 6 colonnes ; et il ne peut pas y avoir plus de 6 villes différentes.

</details>

## 📌 À retenir

| Je veux… | J'écris |
|---|---|
| ouvrir (ou créer) la base | `connexion = sqlite3.connect("sorties/meteolocale.db")` |
| repartir de zéro | `connexion.execute("DROP TABLE IF EXISTS releves")` |
| créer une table | `CREATE TABLE releves (date TEXT, ville TEXT, temp_max REAL, vent_kmh INTEGER)` |
| ajouter une ligne | `connexion.execute("INSERT INTO releves VALUES (?, ?, ?)", (a, b, c))` |
| valider | `connexion.commit()` (sinon : brouillon perdu) |
| lire | `connexion.execute("SELECT date, temp_max FROM releves WHERE ville = ? ORDER BY temp_max DESC", ("Albi",)).fetchall()` |
| corriger | `UPDATE releves SET pluie_mm = ? WHERE ville = ? AND date = ?` + `commit` |
| supprimer | `DELETE FROM releves WHERE ville = ?` + `commit` |
| un DataFrame | `pd.read_sql("SELECT …", connexion, params=(…,))` |
| raccrocher | `connexion.close()` |

**Règle d'or** : une valeur dans une requête = un `?`, jamais une f-string (apostrophes, injection SQL). Un seul `?` :
`(valeur,)` avec la virgule. **Toujours un `WHERE`** avec `UPDATE` et `DELETE`.

Erreurs : `OperationalError … already exists` (table créée deux fois) ; `OperationalError: no such column/table`
(nom mal écrit) ; `OperationalError: near "…": syntax error` (requête mal formée, souvent une f-string) ;
`ProgrammingError: Incorrect number of bindings` (pas autant de valeurs que de `?`).

## 🧠 Mini-quiz

1. Que fait `sqlite3.connect("sorties/x.db")` si le fichier n'existe pas ?
2. Tu as fait 3 `INSERT` et fermé la connexion sans `commit`. Combien de lignes reste-t-il ?
3. Pourquoi `f"... WHERE ville = '{ville}'"` est-il dangereux ? Donne deux raisons.
4. Que renvoie `fetchall()` ?
5. Que se passe-t-il avec `DELETE FROM releves` sans `WHERE` ?

<details><summary>Réponses</summary>

1. Il crée un fichier de base vide, et renvoie la connexion.
2. Aucune nouvelle ligne : sans `commit`, les changements sont un brouillon, perdu à la fermeture.
3. Une apostrophe dans la valeur casse la requête (`syntax error`) ; et une valeur piégée peut modifier la requête
   (injection SQL). Avec `?`, la valeur voyage à part.
4. Une liste de tuples : un tuple par ligne, dans l'ordre des colonnes demandées.
5. Toutes les lignes de la table sont supprimées (après `commit`), sans avertissement.

</details>

## ➡️ Et maintenant

Ouvre `02-exercices.ipynb` : chaque exercice fabrique sa propre petite base dans `sorties/`.